# Waterfalls — what a run actually generates

Two qualitative panels for any discworld run, both built through the one canonical helper
`pim.figures.waterfall_grid` per `research/specs/WATERFALL_SPEC.md` (gray on dark, noisy
context frames above the marked line, **each column its own free-run from step 0**, fixed
0–1 scaling, figure-top legend, ≥3 sample rows).

**§1 — Free-running prediction.** Teacher-force `TEACHER_FORCE` frames, then roll out
`ROLL_STEPS`, against the simulator's clean observations. Answers "does this model
generate a coherent world at all?"

**§2 — Editing.** Teacher-force to the edit frame, apply each canonical editor — **PI,
ND, GS, one column each** — and show the subsequent rollouts against the post-edit ground
truth, with green target / red-dashed ghost locators and each column's Edit Index in its
title. The **unsteered** column is included deliberately: it is where the −1 end of the
index actually sits for this model, and every editor column must be read against it.

Set the run and the knobs in cell [1]; everything else follows. No metric math lives
here — rollouts come from `pim.environments.discworld.bench`, scores from
`pim.metrics`, drawing from `pim.figures`.

In [ ]:
# [1] ── SET THESE ────────────────────────────────────────────────────────────
# RUN         = "noise_ablation/L-dw-noiseless-20m"  # any discworld run under runs/
RUN         = "initial_othello_comparison/L-dw-20m"
INSTANCE    = None      # None = the run's own instance (from its config.json)
N_SAMPLES   = 10        # waterfall rows to draw
TEACHER_FORCE = 20      # §1: frames teacher-forced before the free-run
ROLL_STEPS  = 15        # §1: frames rolled out afterwards
PROBE_SEQS  = 30_000    # probe corpus size for §2's editors (cached per run)
TARGET      = "full"    # ONE probe set per basis, fitted on the full state (2026-09-01)
BASIS       = "frustum" # the canonical discworld basis: u = x/(scale·y), depth 1/y
# §2 uses the edit set's own edit frame (EF=20) and K_ROLL=15 so its Edit Indices are
# directly comparable to every other number in the project — those are NOT knobs here.
# Each editor's DIM SET ("pos" or "all") is not a knob either: it comes from the run's
# scores.json, which reports whichever of the two won.
# ─────────────────────────────────────────────────────────────────────────────
import json, os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO); sys.path.insert(0, str(REPO))

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from pim.models import load_checkpoint
from pim.environments.discworld import bench as dwb
from pim.figures import waterfall_grid

run_dir = REPO / "runs" / RUN
cfg = json.loads((run_dir / "config.json").read_text())
inst = INSTANCE or cfg["data"]["instance"]
inst_root = REPO / "datasets" / "discworld" / inst
eval_dir, probe_corpus = inst_root / "eval", inst_root / "probe"

model, info = load_checkpoint(run_dir / "best_model.pt", device=dwb.DEV)
FIGS = run_dir / "figures"; FIGS.mkdir(exist_ok=True)
print(f"{RUN}\n  {info.arch} · {cfg['n_params']:,} params · val {info.val_loss:.6f}")
print(f"  instance {inst}  (eval {eval_dir.name}/, probes {probe_corpus.name}/)")
print(f"  §1: teacher-force {TEACHER_FORCE} → roll {ROLL_STEPS} · {N_SAMPLES} samples")
print(f"  §2: edit frame {dwb.EF}, roll {dwb.K_ROLL}, target={TARGET}, basis={BASIS}")

## §1 — Free-running prediction vs the simulator

Teacher-force `TEACHER_FORCE` frames of the held-out **test** split, then free-run.
Rollout step 0 is the model's prediction *of* frame `TEACHER_FORCE`, so it aligns with
`clean_obs[TEACHER_FORCE : +ROLL_STEPS]` — no slicing, no dropped step. The context strip
above the line is the **noisy** observation the model was actually fed; only the GT
column is the clean render.

In [ ]:
# [2] §1 — free-run the test split, against the clean GT and as a signed error map.
from pim.environments.discworld.loading import load_dataset

N_CTX = 6            # context frames shown above the line (spec)
DIFF_SCALE = 1.0     # ± range of the error colormap, FIXED across every cell and sample.
                     # Observations live in [0,1] so 1.0 is the true worst case; lower it
                     # to amplify small errors — but then say so when quoting the figure.

test = load_dataset(str(eval_dir), n_obj_keep=2, require_edits=False).test
obs, clean = test.obs[:N_SAMPLES], test.clean_obs[:N_SAMPLES]

roll = dwb.free_rollout(model, obs, TEACHER_FORCE, ROLL_STEPS)
gt = clean[:, TEACHER_FORCE : TEACHER_FORCE + ROLL_STEPS]
err = roll - gt                                   # >0 over-predicts, <0 under-predicts
rmse = float(np.sqrt((err ** 2).mean()))

fig = waterfall_grid(
    columns={"model free-run": roll},
    # metric goes on the prediction column only — the error map's own scale is stated in
    # the title and legend, and labelling it with an RMSE it does not show misleads.
    diff_columns={f"error (pred − GT), ±{DIFF_SCALE:g}": err},
    diff_scale=DIFF_SCALE,
    context=obs[:, TEACHER_FORCE - N_CTX : TEACHER_FORCE],   # the NOISY frames it was fed
    gt=gt,
    title=(f"{RUN} — free-running prediction after {TEACHER_FORCE} teacher-forced frames "
           f"({ROLL_STEPS}-step rollout, {inst})"),
    sample_idx=range(N_SAMPLES),
    metrics={"model free-run": rmse},
    metric_label="obs RMSE",
)
fig.savefig(FIGS / "waterfall_freerun.png", dpi=120, bbox_inches="tight")
display(fig); plt.close(fig)
print(f"rollout obs RMSE {rmse:.4f} | max |error| {np.abs(err).max():.3f} | "
      f"mean signed error {err.mean():+.5f} (≈0 ⇒ drift, not a bias)  →  "
      f"{(FIGS / 'waterfall_freerun.png').relative_to(REPO)}")

## §2 — Editing: each canonical editor, its own column

Teacher-force to the edit frame, write with **PI / ND / GS**, free-run, and compare to
the **post-edit** ground truth. Each editor is shown at its own best arm (point and α
chosen by Edit Index over the canonical sweep), because a fixed arm would flatter
whichever editor happens to like it.

Read every column against **unsteered** — that is where this model's −1 end actually
sits — and read the Edit Index in each title together with the picture: a high index
with a wrecked panel is a destroyed rollout, not an edit.

In [ ]:
# [3] §2a — bench, probes, and each editor's best arm (reuse this run's scores.json if
#      it exists, so the picture is drawn at exactly the arm the table reports).
#      An arm is (residual point, α, dim set): the dim set says whether the edit drove
#      the position read-outs only or the whole state.
b = dwb.load_bench(model, n=max(N_SAMPLES, 32), target=TARGET, basis_name=BASIS,
                   data_dir=eval_dir)
lin = dwb.fit_probes(model, target=TARGET, n_seq=PROBE_SEQS, family="linear",
                     basis_name=BASIS, data_dir=probe_corpus,
                     cache_dir=run_dir / "probes", log=None)
mlp = dwb.fit_probes(model, target=TARGET, n_seq=PROBE_SEQS, family="mlp",
                     basis_name=BASIS, data_dir=probe_corpus,
                     cache_dir=run_dir / "probes", log=None)

def _arm(r):
    return int(r["point"]), float(r["alpha"]), r.get("dims", "all")

sp = run_dir / "scores.json"
if sp.exists():
    best = json.loads(sp.read_text())["bases"][BASIS]["best"]
    ARMS = {ed: _arm(best[ed]) for ed in ("PI", "ND", "GS") if best.get(ed)}
    print("arms from scores.json (same as the master table):")
else:  # no scores yet — sweep the canonical grids here, both dim sets
    n_pts = model.n_layers + 1
    nd, pi, gs = [], [], []
    for dims in ("pos", "all"):
        for e in range(n_pts):
            nd += dwb.nanda_arm(model, b, lin[e][0], e, (0.2, 0.5, 1.0, 2.0), dims=dims)
        pi += dwb.pinv_arm(model, b, lin, (1.0, 5.0, 20.0, 60.0, 175.0), dims=dims)
        gs += dwb.grad_steer_arm(model, b, mlp, range(n_pts), (0.05, 0.2, 0.5), dims=dims)
    ARMS = {ed: _arm(max(recs, key=lambda r: r["edit_index"]))
            for ed, recs in (("ND", nd), ("PI", pi), ("GS", gs))}
    print("no scores.json — swept here:")
for ed, (pt, a, dims) in ARMS.items():
    print(f"  {ed:<3} point {pt}  α {a:g}  dims {dims}")

In [ ]:
# [4] §2b — one rollout per editor, then the panel. Rollouts and scores both come from
#      pim, so the picture and the number can never disagree.
(pi_pt, pi_a, pi_d) = ARMS["PI"]
# (nd_pt, nd_a, nd_d) = ARMS["ND"]
(gs_pt, gs_a, gs_d) = ARMS["GS"]
rolls = {"unsteered": dwb.unsteered_rollout(model, b)}
rolls["PI"] = dwb.pinv_rollout(model, b, lin[pi_pt][0], pi_pt, pi_a,
                               space="zspace", dims=pi_d)
# rolls["ND"] = dwb.nanda_rollout(model, b, lin[nd_pt][0], nd_pt, nd_a, dims=nd_d)
rolls["GS"] = dwb.grad_steer_rollout(model, b, mlp, gs_pt, gs_a, dims=gs_d)

cards = {k: dwb.score(model, b, r) for k, r in rolls.items()}
uns = cards["unsteered"]
ei = {k: c["edit_index"] for k, c in cards.items()}

# ray coordinates of the target / ghost zones, for the green and red-dashed locators
def _cx(mask):
    i = np.where(mask)[0]
    return i.mean() if i.size else np.nan
tgt_x = np.array([_cx(b.zones.target[i]) for i in range(N_SAMPLES)])
gho_x = np.array([_cx(b.zones.ghost[i]) for i in range(N_SAMPLES)])

labels = {k: (k if k == "unsteered"
              else f"{k} ({ARMS[k][2]}·pt{ARMS[k][0]} α{ARMS[k][1]:g})")
          for k in rolls}
fig = waterfall_grid(
    columns={labels[k]: rolls[k][:N_SAMPLES] for k in ("unsteered", "PI", "GS")},
    context=b.obs[:N_SAMPLES, dwb.EF - N_CTX : dwb.EF],      # the NOISY pre-edit frames
    gt=b.gt_roll[:N_SAMPLES],                                # clean POST-EDIT ground truth
    title=(f"{RUN} — canonical editors at the edit frame (EF={dwb.EF}, "
           f"{dwb.K_ROLL}-step rollout, target={TARGET}, basis={BASIS}, {inst})"),
    sample_idx=range(N_SAMPLES),
    target_x=tgt_x, ghost_x=gho_x,
    metrics={labels[k]: ei[k] for k in rolls},
)
fig.savefig(FIGS / "waterfall_edits.png", dpi=120, bbox_inches="tight")
display(fig); plt.close(fig)

print(f"{'editor':<10}{'dims':>6}{'EI':>9}{'fidelity':>10}{'target':>9}{'ghost':>9}"
      f"{'collat':>9}")
for k, c in cards.items():
    fid = 1.0 if k == "unsteered" else dwb.fidelity_ratio(c, uns)
    dims = "—" if k == "unsteered" else ARMS[k][2]
    print(f"{k:<10}{dims:>6}{c['edit_index']:>+9.4f}{fid:>10.3f}{c['target_rmse']:>9.4f}"
          f"{c['ghost_rmse']:>9.4f}{c['collateral_rmse']:>9.4f}")
print("\nfidelity = RMSE(edited, GT) / RMSE(unsteered, GT) at the EDIT STEP; "
      ">1 means the edit left the model further from the post-edit world than "
      "doing nothing.")